# Day 27 — Model Evaluation & Performance Metrics

**30-Day Data Analytics Cookbook**  
**Theme:** Evaluate regression and time-series forecasting models using MAE, MSE, RMSE, MAPE, WAPE and R².

**Learning objectives**
- Calculate and interpret key metrics.
- Compare forecasts with a naive baseline.
- Diagnose bias and residual patterns.
- Apply chronological holdout and rolling-origin validation.
- Understand why metrics must be interpreted alongside operational impact.

> Illustrative synthetic monthly citizen-grievance counts; not official GCC data.

## 1. Setup
Install if needed: `pip install pandas numpy matplotlib scikit-learn`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

## 2. Illustrative complaint forecasts
Both models are evaluated on the **same six observations**.

In [ ]:
df = pd.DataFrame({
    'month': pd.date_range('2026-01-01', periods=6, freq='MS'),
    'actual': [1200, 1350, 1280, 1500, 1420, 1600],
    'model_a': [1150, 1400, 1300, 1450, 1480, 1550],
    'model_b': [1100, 1250, 1200, 1350, 1300, 1450],
})
display(df)

## 3. Define robust evaluation metrics
- **MAE:** mean absolute error in complaints.
- **MSE:** mean squared error (squared complaints).
- **RMSE:** square root of MSE; more sensitive to large errors.
- **MAPE:** average absolute percentage error; undefined for actual values equal to zero.
- **WAPE:** sum of absolute errors / sum of absolute actual values; undefined if denominator is zero.
- **R²:** performance relative to predicting the sample mean; may be negative.
- **Bias:** mean(predicted − actual), identifying systematic over/underprediction.

MAPE and WAPE are reported as percentages.

In [ ]:
def evaluate(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    if actual.shape != predicted.shape or actual.ndim != 1 or len(actual) == 0:
        raise ValueError('Actual and predicted must be nonempty, equally sized 1-D arrays')
    errors = predicted - actual
    nonzero = actual != 0
    return {
        'MAE': mean_absolute_error(actual, predicted),
        'MSE': mean_squared_error(actual, predicted),
        'RMSE': np.sqrt(mean_squared_error(actual, predicted)),
        'MAPE (%)': np.mean(np.abs(errors[nonzero] / actual[nonzero])) * 100 if nonzero.all() else np.nan,
        'WAPE (%)': np.sum(np.abs(errors)) / np.sum(np.abs(actual)) * 100 if np.sum(np.abs(actual)) > 0 else np.nan,
        'R2': r2_score(actual, predicted) if len(actual) > 1 else np.nan,
        'Bias': np.mean(errors),
    }

results = pd.DataFrame({
    'Model A': evaluate(df.actual, df.model_a),
    'Model B': evaluate(df.actual, df.model_b),
}).T
display(results.round(2))
assert results.loc['Model A', 'MAE'] < results.loc['Model B', 'MAE']

## 4. Compare observed and predicted counts

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for col, label in [('actual','Actual'), ('model_a','Model A'), ('model_b','Model B')]:
    ax.plot(df.month, df[col], marker='o', label=label)
ax.set(title='Monthly complaints: actual vs predictions', xlabel='Month', ylabel='Complaints')
ax.legend()
ax.grid(alpha=0.3)
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 5. Residual diagnostics
Residual = actual − predicted. A pattern can indicate systematic bias, trend, or seasonality the model missed.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for col, label in [('model_a', 'Model A'), ('model_b', 'Model B')]:
    ax.plot(df.month, df.actual - df[col], marker='o', label=label)
ax.axhline(0, linestyle='--')
ax.set(title='Forecast residuals', xlabel='Month', ylabel='Actual − predicted')
ax.legend()
ax.grid(alpha=0.3)
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 6. Chronological holdout and naive baseline
Never shuffle time-series observations for forecasting evaluation. Train on earlier months, test on later months. The last-value naive forecast predicts that the next month equals the previous observed month.

In [ ]:
rng = np.random.default_rng(27)
months = pd.date_range('2023-01-01', periods=36, freq='MS')
t = np.arange(len(months))
counts = np.rint(900 + 12*t + 115*np.sin(2*np.pi*t/12) + rng.normal(0, 35, len(t))).astype(int)
series = pd.Series(counts, index=months, name='complaints')
train, test = series.iloc[:30], series.iloc[30:]
# Walk-forward one-step naive baseline: uses the prior *observed* value at each step.
naive = series.shift(1).loc[test.index]
print(f'Train: {train.index.min().date()} to {train.index.max().date()}')
print(f'Test:  {test.index.min().date()} to {test.index.max().date()}')
display(pd.DataFrame({'actual': test, 'naive_forecast': naive}))
display(pd.DataFrame([evaluate(test.values, naive.values)], index=['Naive baseline']).round(2))

## 7. Rolling-origin backtesting
This example evaluates one-step-ahead forecasts using only the immediately preceding observation. A production model should be retrained at each origin when appropriate, and all feature engineering must use only data available at that origin.

In [ ]:
records = []
for i in range(12, len(series)):
    history = series.iloc[:i]
    actual = series.iloc[i]
    predicted = history.iloc[-1]
    records.append({'month': series.index[i], 'actual': actual, 'forecast': predicted})
backtest = pd.DataFrame(records).set_index('month')
print('Rolling-origin, one-step naive forecast')
display(pd.DataFrame([evaluate(backtest.actual, backtest.forecast)], index=['Naive']).round(2))

## 8. Operational interpretation
- **Staffing:** underforecasting complaint volume may cause backlogs.
- **Seasonality:** evaluate monsoon months separately if errors are especially costly.
- **Bias:** inspect whether the model persistently underpredicts peak demand.
- **Model choice:** prefer models that improve on a realistic baseline over several holdout periods.
- **Caution:** the six-month model comparison is educational, not evidence of production readiness.

**Discussion:** When might a slightly higher MAE be acceptable if a model is better at predicting high-demand months?

## 9. Practice exercises
1. Add a third model with a constant 1,400-complaint forecast. Compare its MAE and RMSE.
2. Introduce one very large prediction error and observe the effect on RMSE.
3. Test the metrics with an actual value of zero; explain why MAPE becomes undefined.
4. Calculate MAE for the last six months of the rolling-origin backtest.
5. Explain why a random train/test split could cause leakage in forecasting.

**Key takeaway:** The best model is not the most complicated one; it is the one that improves on a baseline and supports reliable decisions.

**Repository:** https://github.com/krishnakumarimv/30-Days-Data-Analytics-Cookbook